In [0]:
%run ./00_mounts

configured


In [0]:
from pyspark.sql.functions import current_date, lit, col, row_number, expr
from pyspark.sql.window import Window
from delta.tables import DeltaTable

base = f"{SRC}/csv"
y = max(f.name.strip("/") for f in dbutils.fs.ls(base))
m = max(f.name.strip("/") for f in dbutils.fs.ls(f"{base}/{y}"))
d = max(f.name.strip("/") for f in dbutils.fs.ls(f"{base}/{y}/{m}"))
latest_path = f"{base}/{y}/{m}/{d}/*.parquet"
file_date = f"{y}-{m}-{d}"

inc = (spark.read.parquet(latest_path)
       .withColumn("payment_dim_id", expr("uuid()"))
       .withColumn("is_active", lit(1))
       .withColumn("start_date", current_date())
       .withColumn("end_date", lit(None).cast("date"))
       .withColumn("ingest_date", current_date())
       .withColumn("file_date", lit(file_date)))
inc.printSchema()   # cols = PL4_1 mapping sink names (orderid, paymentsequential, ...)

w = Window.partitionBy("orderid","paymentsequential").orderBy(col("ingest_date").desc())
inc = inc.withColumn("rn", row_number().over(w)).filter("rn=1").drop("rn")

delta_path = f"{STD}/orderpay_scd2_delta"
if not DeltaTable.isDeltaTable(spark, delta_path):
    inc.write.format("delta").mode("overwrite").option("mergeSchema","true").save(delta_path)
else:
    (DeltaTable.forPath(spark, delta_path).alias("t")
     .merge(inc.alias("s"), "t.orderid=s.orderid AND t.paymentsequential=s.paymentsequential")
     .whenMatchedUpdate(
        condition="t.is_active=1 AND (t.paymenttype<>s.paymenttype OR t.paymentinstallments<>s.paymentinstallments OR t.paymentvalue<>s.paymentvalue)",
        set={"is_active": lit(0), "end_date": current_date()})
     .whenNotMatchedInsertAll().execute())



root
 |-- orderid: string (nullable = true)
 |-- paymentsequential: string (nullable = true)
 |-- paymenttype: string (nullable = true)
 |-- paymentinstallments: string (nullable = true)
 |-- paymentvalue: string (nullable = true)
 |-- payment_dim_id: string (nullable = false)
 |-- is_active: integer (nullable = false)
 |-- start_date: date (nullable = false)
 |-- end_date: date (nullable = true)
 |-- ingest_date: date (nullable = false)
 |-- file_date: string (nullable = false)



---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-4622236769152768>, line 35
     27 else:
     28     (DeltaTable.forPath(spark, delta_path).alias("t")
     29      .merge(inc.alias("s"), "t.orderid=s.orderid AND t.paymentsequential=s.paymentsequential")
     30      .whenMatchedUpdate(
     31         condition="t.is_active=1 AND (t.paymenttype<>s.paymenttype OR t.paymentinstallments<>s.paymentinstallments OR t.paymentvalue<>s.paymentvalue)",
     32         set={"is_active": lit(0), "end_date": current_date()})
     33      .whenNotMatchedInsertAll().execute())
---> 35 spark.sql(f"CREATE TABLE IF NOT EXISTS orderpay_scd2_delta USING DELTA LOCATION '{delta_path}'")

File /databricks/spark/python/pyspark/sql/connect/session.py:879, in SparkSession.sql(self, sqlQuery, args, **kwargs)
    876         _views.append(SubqueryAlias(df._plan, name))
    878 cmd = SQL(sqlQuery, 

In [0]:
STD = "abfss://input@bhumeetadlsdevstd01.dfs.core.windows.net"
spark.sql(f"SELECT COUNT(*) AS total FROM delta.`{STD}/orderpay_scd2_delta`").show()
spark.sql(f"SELECT is_active, COUNT(*) AS cnt FROM delta.`{STD}/orderpay_scd2_delta` GROUP BY is_active").show()